# MCI-GRU Experiment Workflow - Google Colab

Complete workflow for running MCI-GRU experiments with persistent Google Drive storage.

**Features:**
- Configurable data universe (different CSV datasets)
- Configurable loss function (MSE, IC, or Combined MSE+IC)
- Configurable date ranges for train/val/test splits
- Automatic output organization with timestamps
- Comprehensive backtest with optional transaction costs
- All outputs synced to Google Drive

## 1. Setup: Mount Google Drive and Install Dependencies

In [ ]:
from google.colab import drive
import os
from datetime import datetime

# Mount Google Drive
drive.mount('/content/drive')

# Setup experiment directory in Google Drive
GDRIVE_BASE = '/content/drive/MyDrive/MCI-GRU-Experiments'
os.makedirs(GDRIVE_BASE, exist_ok=True)

print(f"Google Drive mounted")
print(f"Experiment directory: {GDRIVE_BASE}")
print(f"All outputs will be automatically saved here")

In [ ]:
# Clone repository (if not already done)
REPO_URL = 'https://github.com/magilliam27/MCI-GRU.git'
BRANCH = 'lC_loss'  # Change to your working branch

if not os.path.exists('/content/MCI-GRU'):
    !git clone -b {BRANCH} {REPO_URL} /content/MCI-GRU
    print(f"Repository cloned (branch: {BRANCH})")
else:
    %cd /content/MCI-GRU
    !git fetch origin
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
    print(f"Repository updated (branch: {BRANCH})")

# Change to project directory
%cd /content/MCI-GRU

# Install requirements
!pip install -q -r requirements.txt
print("Dependencies installed")

## 2. Experiment Configuration

Set ALL experiment parameters in one place. Change these values and re-run
the cells below to run a new experiment.

In [ ]:
# ============================================================
#  EXPERIMENT CONFIGURATION - Edit these values
# ============================================================

# --- Data Universe ---
# Which CSV dataset to use. Upload your file to /content/MCI-GRU/
# Examples: 'sp500_2019_universe_data.csv', 'sp500_2016_universe_data.csv'
DATA_FILE = 'sp500_2019_universe_data.csv'

# --- Date Ranges ---
TRAIN_START = '2019-01-01'
TRAIN_END   = '2023-12-31'
VAL_START   = '2024-01-01'
VAL_END     = '2024-12-31'
TEST_START  = '2025-01-01'
TEST_END    = '2025-12-31'

# --- Loss Function ---
# Options: 'mse', 'ic', 'combined'
#   mse      = Standard MSE loss (paper baseline)
#   ic       = Pure IC loss (negative Pearson correlation)
#   combined = (1-alpha)*MSE + alpha*(-IC)
LOSS_TYPE = 'combined'
IC_LOSS_ALPHA = 0.5  # Only used when LOSS_TYPE='combined' (0.0 to 1.0)

# --- Training ---
NUM_MODELS = 10        # Number of models to train and average
NUM_EPOCHS = 100       # Max epochs per model
LEARNING_RATE = 0.0002
BATCH_SIZE = 32
EARLY_STOPPING = 10   # Patience for early stopping

# --- Experiment Name ---
# Used for output directory naming. Change for each experiment.
EXPERIMENT_NAME = f'{LOSS_TYPE}_alpha{int(IC_LOSS_ALPHA*100)}_{DATA_FILE.replace(".csv", "")}'

# --- Backtesting ---
TOP_K = 10             # Number of top stocks in portfolio (used in Section 6)
TOP_K_VALUES = [10, 20, 30, 40, 50]  # For Top-K sweep (Section 6b)
TRANSACTION_COSTS = True
SPREAD_BPS = 10        # Bid-ask spread in basis points
SLIPPAGE_BPS = 5       # Slippage in basis points

# ============================================================
#  Print configuration summary
# ============================================================
print('=' * 70)
print('EXPERIMENT CONFIGURATION')
print('=' * 70)
print(f'  Data file:       {DATA_FILE}')
print(f'  Train:           {TRAIN_START} to {TRAIN_END}')
print(f'  Validation:      {VAL_START} to {VAL_END}')
print(f'  Test:            {TEST_START} to {TEST_END}')
print(f'  Loss:            {LOSS_TYPE}' + (f' (alpha={IC_LOSS_ALPHA})' if LOSS_TYPE == 'combined' else ''))
print(f'  Models:          {NUM_MODELS}')
print(f'  Epochs:          {NUM_EPOCHS}')
print(f'  Experiment name: {EXPERIMENT_NAME}')
print(f'  Top-K:           {TOP_K}')
print(f'  Top-K sweep:     {TOP_K_VALUES}')
print(f'  Costs:           {"ON" if TRANSACTION_COSTS else "OFF"}' + (f' ({SPREAD_BPS}bps spread, {SLIPPAGE_BPS}bps slip)' if TRANSACTION_COSTS else ''))
print('=' * 70)

## 3. Data Preparation

Upload your data CSV to `/content/MCI-GRU/` or copy from Google Drive.

In [ ]:
import os
import pandas as pd

data_path = f'/content/MCI-GRU/{DATA_FILE}'

# Option 1: Check if data file already exists in repo
if os.path.exists(data_path):
    print(f'Found: {DATA_FILE}')
else:
    # Option 2: Try copying from Google Drive
    gdrive_data = f'/content/drive/MyDrive/MCI-GRU-Data/{DATA_FILE}'
    if os.path.exists(gdrive_data):
        !cp "{gdrive_data}" "{data_path}"
        print(f'Copied from Google Drive: {DATA_FILE}')
    else:
        print(f'MISSING: {DATA_FILE}')
        print(f'  Option A: Upload to /content/MCI-GRU/{DATA_FILE}')
        print(f'  Option B: Place in Google Drive at MCI-GRU-Data/{DATA_FILE}')
        print(f'  Option C: Use Colab file upload (uncomment cell below)')

# Verify data coverage
if os.path.exists(data_path):
    df = pd.read_csv(data_path, usecols=['dt', 'kdcode'])
    df['dt'] = pd.to_datetime(df['dt'])
    print(f'  Rows:    {len(df):,}')
    print(f'  Stocks:  {df["kdcode"].nunique()}')
    print(f'  Dates:   {df["dt"].min().date()} to {df["dt"].max().date()}')

    # Check coverage against configured date ranges
    data_min, data_max = str(df['dt'].min().date()), str(df['dt'].max().date())
    if data_min > TRAIN_START:
        print(f'  WARNING: Data starts at {data_min}, but TRAIN_START is {TRAIN_START}')
    if data_max < TEST_END:
        print(f'  WARNING: Data ends at {data_max}, but TEST_END is {TEST_END}')
    del df

In [ ]:
# Uncomment to upload data file manually via Colab upload dialog
# from google.colab import files
# uploaded = files.upload()
# for fn in uploaded:
#     !mv "{fn}" /content/MCI-GRU/
#     print(f'Uploaded: {fn}')

## 4. Run Training

Trains models using the configuration from Section 2.

In [ ]:
!python run_experiment.py \
    data.source=csv \
    data.filename={DATA_FILE} \
    data.train_start={TRAIN_START} \
    data.train_end={TRAIN_END} \
    data.val_start={VAL_START} \
    data.val_end={VAL_END} \
    data.test_start={TEST_START} \
    data.test_end={TEST_END} \
    training.loss_type={LOSS_TYPE} \
    training.ic_loss_alpha={IC_LOSS_ALPHA} \
    training.num_models={NUM_MODELS} \
    training.num_epochs={NUM_EPOCHS} \
    training.learning_rate={LEARNING_RATE} \
    training.batch_size={BATCH_SIZE} \
    training.early_stopping_patience={EARLY_STOPPING} \
    output_dir={GDRIVE_BASE} \
    experiment_name={EXPERIMENT_NAME}

print('\n' + '=' * 80)
print('Training complete!')
print(f'Check outputs in: {GDRIVE_BASE}/{EXPERIMENT_NAME}/')
print('=' * 80)

## 5. Find Latest Experiment Run

In [ ]:
import glob
import os

def find_latest_run(experiment_name):
    """Find the most recent run for an experiment."""
    experiment_dir = f"{GDRIVE_BASE}/{experiment_name}"

    # Look for timestamped directories
    run_dirs = sorted(glob.glob(f"{experiment_dir}/*/"))

    if run_dirs:
        latest_run = run_dirs[-1]
        print(f"Latest run: {latest_run}")
        return latest_run
    elif os.path.exists(experiment_dir):
        print(f"Using experiment directory: {experiment_dir}")
        return experiment_dir + '/'
    else:
        print(f"No runs found for experiment: {experiment_name}")
        return None

# Find latest run for the current experiment
LATEST_RUN = find_latest_run(EXPERIMENT_NAME)

if LATEST_RUN:
    PREDICTIONS_PATH = os.path.join(LATEST_RUN, 'averaged_predictions')
    print(f"\nPredictions path: {PREDICTIONS_PATH}")

    # List generated files
    if os.path.exists(LATEST_RUN):
        print(f"\nGenerated files:")
        !ls -lh {LATEST_RUN}

## 6. Run Backtesting

Evaluate predictions against actual market data. Uses the same DATA_FILE
and date ranges from Section 2.

In [ ]:
# Backtest WITHOUT transaction costs
print('Running backtest WITHOUT transaction costs...\n')

!python tests/backtest_sp500.py \
    --predictions_dir {PREDICTIONS_PATH} \
    --data_file {DATA_FILE} \
    --test_start {TEST_START} \
    --test_end {TEST_END} \
    --top_k {TOP_K} \
    --auto_save \
    --plot

print(f'\nBacktest complete (no costs)')
print(f'Results: {LATEST_RUN}backtest/')

In [ ]:
# Backtest WITH transaction costs
if TRANSACTION_COSTS:
    print('Running backtest WITH transaction costs...\n')

    !python tests/backtest_sp500.py \
        --predictions_dir {PREDICTIONS_PATH} \
        --data_file {DATA_FILE} \
        --test_start {TEST_START} \
        --test_end {TEST_END} \
        --top_k {TOP_K} \
        --auto_save \
        --backtest_suffix _with_costs \
        --transaction_costs \
        --spread {SPREAD_BPS} \
        --slippage {SLIPPAGE_BPS} \
        --plot

    print(f'\nBacktest complete (with costs)')
    print(f'Results: {LATEST_RUN}backtest_with_costs/')
else:
    print('Transaction costs disabled - skipping')

## 6b. Top-K Sweep (10, 20, 30, 40, 50)

Run backtests for top-k = 10, 20, 30, 40, 50 to see how adding more stocks changes returns. 
**All results are saved to Google Drive** (same experiment directory as training outputs).

In [ ]:
# Top-K Sweep: all results save to Google Drive (LATEST_RUN is under GDRIVE_BASE)
for k in TOP_K_VALUES:
    print(f"\n{'='*70}")
    print(f"Backtesting TOP-K = {k} (saving to Google Drive)")
    print('='*70)

    cmd = f"""python tests/backtest_sp500.py \
        --predictions_dir {PREDICTIONS_PATH} \
        --data_file {DATA_FILE} \
        --test_start {TEST_START} \
        --test_end {TEST_END} \
        --top_k {k} \
        --auto_save \
        --backtest_suffix _topk{k}"""
    
    if TRANSACTION_COSTS:
        cmd += f" --transaction_costs --spread {SPREAD_BPS} --slippage {SLIPPAGE_BPS}"
    
    cmd += " --plot"
    get_ipython().system(cmd)

print(f'\n{"="*70}')
print('Top-K sweep complete! Results saved to Google Drive:')
print(f'  {LATEST_RUN}')
for k in TOP_K_VALUES:
    print(f'    backtest_topk{k}/')
print('='*70)

In [ ]:
# Load and compare Top-K sweep results (all from Google Drive)
# Sweep saves to backtest_topk{k}/ (transaction costs applied based on config)
import pandas as pd

topk_results = []
for k in TOP_K_VALUES:
    results_file = os.path.join(LATEST_RUN, f'backtest_topk{k}', 'backtest_results.csv')
    if os.path.exists(results_file):
        df = pd.read_csv(results_file)
        df['top_k'] = k
        topk_results.append(df)
        print(f"  Loaded: top_k={k}")

if topk_results:
    topk_df = pd.concat(topk_results, ignore_index=True)
    key_cols = ['top_k', 'ARR', 'AVoL', 'ASR', 'MDD', 'CR', 'IR']
    available = [c for c in key_cols if c in topk_df.columns]
    display(topk_df[available])

    # Save comparison to Google Drive
    out_file = os.path.join(LATEST_RUN, 'topk_sweep_comparison.csv')
    topk_df.to_csv(out_file, index=False)
    print(f'\nComparison saved to Google Drive: {out_file}')
else:
    print("No results found. Run the Top-K sweep cell above first.")

## 7. View Backtest Results

In [ ]:
# View summary text file
suffix = '_with_costs' if TRANSACTION_COSTS else ''
summary_file = os.path.join(LATEST_RUN, f'backtest{suffix}', 'summary.txt')
if os.path.exists(summary_file):
    with open(summary_file, 'r') as f:
        print(f.read())
else:
    print(f'Summary file not found: {summary_file}')

In [ ]:
# Load and display results DataFrame
import pandas as pd

results_file = os.path.join(LATEST_RUN, f'backtest{suffix}', 'backtest_results.csv')
if os.path.exists(results_file):
    results_df = pd.read_csv(results_file)
    display(results_df.T)  # Transpose for easier reading
else:
    print('Results file not found')

In [ ]:
# Display equity curve
from IPython.display import Image, display as ipy_display

equity_plot = os.path.join(LATEST_RUN, f'backtest{suffix}', 'equity_curve.png')
if os.path.exists(equity_plot):
    ipy_display(Image(filename=equity_plot))
else:
    print('Equity curve plot not found')

## 8. Compare Multiple Experiments

After running several experiments (with different loss types, alphas, or
data universes), list their names here to compare results side by side.

In [ ]:
import pandas as pd
import os

def load_experiment_results(experiment_name, with_costs=False):
    """Load backtest results from an experiment."""
    latest_run = find_latest_run(experiment_name)
    if not latest_run:
        return None

    cost_suffix = '_with_costs' if with_costs else ''
    results_file = os.path.join(latest_run, f'backtest{cost_suffix}', 'backtest_results.csv')

    if os.path.exists(results_file):
        df = pd.read_csv(results_file)
        df['experiment'] = experiment_name
        df['transaction_costs'] = with_costs
        return df
    return None

# ============================================================
#  LIST YOUR EXPERIMENTS TO COMPARE
# ============================================================
experiments = [
    # Add experiment names here after running them.
    # Examples:
    # 'mse_alpha50_sp500_2019_universe_data',
    # 'combined_alpha50_sp500_2019_universe_data',
    # 'ic_alpha50_sp500_2019_universe_data',
    # 'combined_alpha70_sp500_2019_universe_data',
    EXPERIMENT_NAME,  # Current experiment
]

all_results = []

print('Loading experiment results...\n')
for exp in experiments:
    for with_costs in [False, True]:
        result = load_experiment_results(exp, with_costs=with_costs)
        if result is not None:
            all_results.append(result)
            tag = 'with costs' if with_costs else 'no costs'
            print(f'  Found: {exp} ({tag})')

if all_results:
    comparison_df = pd.concat(all_results, ignore_index=True)

    # Select key metrics
    key_metrics = ['experiment', 'transaction_costs',
                   'ARR', 'AVoL', 'ASR', 'MDD', 'CR', 'IR', 'MSE', 'MAE']
    available = [c for c in key_metrics if c in comparison_df.columns]
    comparison_df = comparison_df[available]

    print('\n' + '=' * 100)
    print('EXPERIMENT COMPARISON')
    print('=' * 100)
    display(comparison_df)

    # Save comparison
    comparison_file = f'{GDRIVE_BASE}/experiment_comparison.csv'
    comparison_df.to_csv(comparison_file, index=False)
    print(f'\nComparison saved to: {comparison_file}')
else:
    print('No results found to compare')

## 9. Visualize Performance Comparison

In [ ]:
import matplotlib.pyplot as plt

if all_results and len(comparison_df) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    metrics_to_plot = [
        ('ARR', 'Annualized Return (ARR)', axes[0, 0]),
        ('ASR', 'Annualized Sharpe Ratio (ASR)', axes[0, 1]),
        ('MDD', 'Maximum Drawdown (MDD)', axes[1, 0]),
        ('IR', 'Information Ratio (IR)', axes[1, 1]),
    ]

    for metric, title, ax in metrics_to_plot:
        if metric in comparison_df.columns:
            comparison_df.plot(x='experiment', y=metric, kind='bar', ax=ax, legend=False)
            ax.set_title(title)
            ax.set_ylabel(metric)
            ax.axhline(y=0, color='r', linestyle='--', alpha=0.3)

    plt.tight_layout()

    comparison_plot = f'{GDRIVE_BASE}/experiment_comparison.png'
    plt.savefig(comparison_plot, dpi=300, bbox_inches='tight')
    plt.show()

    print(f'Comparison plot saved to: {comparison_plot}')

## 10. View Daily Returns Time Series

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

returns_file = os.path.join(LATEST_RUN, f'backtest{suffix}', 'daily_returns.csv')

if os.path.exists(returns_file):
    returns_df = pd.read_csv(returns_file)
    returns_df['date'] = pd.to_datetime(returns_df['date'])

    # Plot cumulative returns
    returns_df['cum_portfolio'] = (1 + returns_df['portfolio_return']).cumprod()
    returns_df['cum_benchmark'] = (1 + returns_df['benchmark_return']).cumprod()

    fig, axes = plt.subplots(2, 1, figsize=(15, 10))

    # Cumulative returns
    ax1 = axes[0]
    ax1.plot(returns_df['date'], returns_df['cum_portfolio'], label='Portfolio', linewidth=2)
    ax1.plot(returns_df['date'], returns_df['cum_benchmark'], label='Benchmark', linewidth=2)
    ax1.set_title('Cumulative Returns', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Cumulative Return')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Daily returns
    ax2 = axes[1]
    ax2.plot(returns_df['date'], returns_df['portfolio_return'], alpha=0.7, label='Portfolio Daily Return')
    ax2.axhline(y=0, color='r', linestyle='--', alpha=0.3)
    ax2.set_title('Daily Returns', fontsize=14, fontweight='bold')
    ax2.set_ylabel('Daily Return')
    ax2.set_xlabel('Date')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Display statistics
    print('\nDaily Returns Statistics:')
    print('=' * 50)
    print(returns_df[['portfolio_return', 'benchmark_return', 'excess_return']].describe())
else:
    print('Daily returns file not found')

## 11. Quick A/B Test: Loss Function Comparison

Convenience section to run MSE baseline vs Combined loss on the same data
and compare results. Adjust `num_models` for faster iteration.

In [ ]:
# --- A/B Test Configuration ---
AB_NUM_MODELS = 3   # Use fewer models for faster iteration
AB_NUM_EPOCHS = 50  # Fewer epochs for quick test

ab_experiments = [
    {'name': 'ab_mse_baseline',    'loss_type': 'mse',      'alpha': 0.5},
    {'name': 'ab_combined_a50',    'loss_type': 'combined',  'alpha': 0.5},
    {'name': 'ab_combined_a70',    'loss_type': 'combined',  'alpha': 0.7},
    {'name': 'ab_ic_pure',         'loss_type': 'ic',        'alpha': 0.5},
]

for exp in ab_experiments:
    print(f"\n{'='*80}")
    print(f"Running: {exp['name']} (loss={exp['loss_type']}, alpha={exp['alpha']})")
    print('=' * 80)

    !python run_experiment.py \
        data.source=csv \
        data.filename={DATA_FILE} \
        data.train_start={TRAIN_START} \
        data.train_end={TRAIN_END} \
        data.val_start={VAL_START} \
        data.val_end={VAL_END} \
        data.test_start={TEST_START} \
        data.test_end={TEST_END} \
        training.loss_type={exp['loss_type']} \
        training.ic_loss_alpha={exp['alpha']} \
        training.num_models={AB_NUM_MODELS} \
        training.num_epochs={AB_NUM_EPOCHS} \
        output_dir={GDRIVE_BASE} \
        experiment_name={exp['name']}

print('\nAll A/B experiments complete!')

In [ ]:
# Run backtests for all A/B experiments
for exp in ab_experiments:
    run_dir = find_latest_run(exp['name'])
    if run_dir is None:
        continue
    pred_path = os.path.join(run_dir, 'averaged_predictions')
    if not os.path.exists(pred_path):
        print(f"  No predictions for {exp['name']}, skipping")
        continue

    print(f"\nBacktesting: {exp['name']}")
    !python tests/backtest_sp500.py \
        --predictions_dir {pred_path} \
        --data_file {DATA_FILE} \
        --test_start {TEST_START} \
        --test_end {TEST_END} \
        --top_k {TOP_K} \
        --auto_save \
        --backtest_suffix _with_costs \
        --transaction_costs \
        --spread {SPREAD_BPS} \
        --slippage {SLIPPAGE_BPS}

print('\nAll backtests complete!')

In [ ]:
# Compare A/B test results
ab_names = [exp['name'] for exp in ab_experiments]
ab_results = []

print('Loading A/B test results...\n')
for name in ab_names:
    result = load_experiment_results(name, with_costs=True)
    if result is not None:
        ab_results.append(result)
        print(f'  Found: {name}')

if ab_results:
    ab_df = pd.concat(ab_results, ignore_index=True)
    key_metrics = ['experiment', 'ARR', 'AVoL', 'ASR', 'MDD', 'CR', 'IR', 'MSE', 'MAE']
    available = [c for c in key_metrics if c in ab_df.columns]

    print('\n' + '=' * 100)
    print('A/B TEST COMPARISON: Loss Function Impact')
    print('=' * 100)
    display(ab_df[available])

    ab_file = f'{GDRIVE_BASE}/ab_loss_comparison.csv'
    ab_df[available].to_csv(ab_file, index=False)
    print(f'\nSaved to: {ab_file}')
else:
    print('No A/B results found')

## 12. Access Files from Google Drive

All outputs are saved to Google Drive and accessible from any device.

In [ ]:
# List all experiment outputs
print('Your experiment outputs in Google Drive:')
print('=' * 80)
!ls -lh {GDRIVE_BASE}

print('\n' + '=' * 80)
print('Access these files from Google Drive at:')
print('https://drive.google.com/drive/folders/ (navigate to MCI-GRU-Experiments)')
print('=' * 80)